# 🧠 Stage 1: Matrix Encoding Experiments for ARC

## Overview
This notebook implements and compares **two matrix encoding approaches** for ARC tasks:
- **Experiment 1A**: Embedding + CNN Encoder
- **Experiment 1B**: One-Hot + CNN Encoder

We'll test reconstruction accuracy, feature quality, and computational efficiency on real ARC data.

## Goals
- Establish optimal method for encoding integer matrices (0-9)
- Compare semantic embeddings vs categorical one-hot encoding
- Measure reconstruction accuracy >95%
- Select best encoder for Stage 2 experiments

In [ ]:
# Import required libraries
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import time
import warnings
from tqdm import tqdm  # Add this import
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set matplotlib style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Load ARC dataset from Kaggle input
def load_arc_data():
    """Load ARC training data from Kaggle input directory"""
    
    # Kaggle data paths
    train_challenges_path = '/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json'
    train_solutions_path = '/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json'
    
    try:
        # Load training challenges
        with open(train_challenges_path, 'r') as f:
            challenges = json.load(f)
        
        # Load training solutions
        with open(train_solutions_path, 'r') as f:
            solutions = json.load(f)
        
        print(f"Loaded {len(challenges)} training tasks")
        print(f"Loaded {len(solutions)} training solutions")
        
        return challenges, solutions
    
    except FileNotFoundError:
        print("Kaggle data not found. Using local test data...")
        # Create some test data for local development
        test_data = create_test_data()
        return test_data, test_data

def create_test_data():
    """Create synthetic test data for local development"""
    test_tasks = {}
    
    for i in range(20):  # Create 20 test tasks
        task_id = f"test_{i:03d}"
        
        # Create simple patterns for testing
        if i % 4 == 0:  # Color replacement
            input_matrix = np.random.randint(0, 5, (8, 8))
            output_matrix = np.where(input_matrix == 1, 7, input_matrix)
        elif i % 4 == 1:  # Mirror horizontal
            input_matrix = np.random.randint(0, 5, (6, 8))
            output_matrix = np.fliplr(input_matrix)
        elif i % 4 == 2:  # Rotation 90
            input_matrix = np.random.randint(0, 5, (6, 6))
            output_matrix = np.rot90(input_matrix)
        else:  # Simple fill
            input_matrix = np.random.randint(0, 3, (5, 5))
            output_matrix = np.full_like(input_matrix, 8)
        
        test_tasks[task_id] = {
            'train': [
                {
                    'input': input_matrix.tolist(),
                    'output': output_matrix.tolist()
                }
            ],
            'test': [
                {
                    'input': input_matrix.tolist(),
                    'output': output_matrix.tolist()
                }
            ]
        }
    
    return test_tasks

# Load the data
challenges, solutions = load_arc_data()

# Sample a task to examine structure
sample_task_id = list(challenges.keys())[0]
sample_task = challenges[sample_task_id]

print(f"\nSample Task ID: {sample_task_id}")
print(f"Number of training examples: {len(sample_task['train'])}")
print(f"Number of test examples: {len(sample_task['test'])}")

# Show first training example
first_example = sample_task['train'][0]
input_grid = np.array(first_example['input'])
output_grid = np.array(first_example['output'])

print(f"\nFirst training example:")
print(f"Input shape: {input_grid.shape}")
print(f"Output shape: {output_grid.shape}")
print(f"Input unique values: {np.unique(input_grid)}")
print(f"Output unique values: {np.unique(output_grid)}")

# Visualize the first example
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(input_grid, cmap='tab10', vmin=0, vmax=9)
ax1.set_title('Input Grid')
ax1.axis('off')

im2 = ax2.imshow(output_grid, cmap='tab10', vmin=0, vmax=9)
ax2.set_title('Output Grid')
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Dataset class for ARC matrices
class ARCMatrixDataset(Dataset):
    """Dataset class for ARC matrix reconstruction tasks"""
    
    def __init__(self, challenges, solutions=None, max_size=30):
        self.matrices = []
        self.task_ids = []
        self.max_size = max_size
        
        # Extract all matrices from challenges
        for task_id, task_data in challenges.items():
            # Add training examples
            for example in task_data['train']:
                matrix = np.array(example['input'])
                self.matrices.append(matrix)
                self.task_ids.append(f"{task_id}_train_input")
                
                matrix = np.array(example['output'])
                self.matrices.append(matrix)
                self.task_ids.append(f"{task_id}_train_output")
            
            # Add test examples (input only for challenges)
            for i, example in enumerate(task_data['test']):
                matrix = np.array(example['input'])
                self.matrices.append(matrix)
                self.task_ids.append(f"{task_id}_test_{i}_input")
                
                # Add test output if available (from solutions)
                if solutions and task_id in solutions:
                    if i < len(solutions[task_id]):
                        matrix = np.array(solutions[task_id][i])
                        self.matrices.append(matrix)
                        self.task_ids.append(f"{task_id}_test_{i}_output")
        
        print(f"Created dataset with {len(self.matrices)} matrices")
        
        # Analyze dataset statistics
        self._analyze_dataset()
    
    def _analyze_dataset(self):
        """Analyze dataset statistics"""
        shapes = [matrix.shape for matrix in self.matrices]
        heights = [shape[0] for shape in shapes]
        widths = [shape[1] for shape in shapes]
        
        print(f"Matrix size statistics:")
        print(f"  Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.1f}")
        print(f"  Width: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.1f}")
        
        # Color distribution
        all_values = []
        for matrix in self.matrices:
            all_values.extend(matrix.flatten())
        
        unique_values, counts = np.unique(all_values, return_counts=True)
        print(f"Color distribution: {dict(zip(unique_values, counts))}")
    
    def __len__(self):
        return len(self.matrices)
    
    def __getitem__(self, idx):
        matrix = self.matrices[idx]
        task_id = self.task_ids[idx]
        
        # Pad to max_size if needed
        if matrix.shape[0] > self.max_size or matrix.shape[1] > self.max_size:
            # Crop if larger than max_size
            matrix = matrix[:self.max_size, :self.max_size]
        
        # Pad to max_size
        padded_matrix = np.zeros((self.max_size, self.max_size), dtype=np.int64)
        h, w = matrix.shape
        padded_matrix[:h, :w] = matrix
        
        return torch.tensor(padded_matrix, dtype=torch.long), task_id, (h, w)

# Create dataset
dataset = ARCMatrixDataset(challenges, solutions)

# Create data loader
batch_size = 16
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\nDataLoader created with batch size {batch_size}")
print(f"Total batches: {len(dataloader)}")

# Test the dataloader
for batch_matrices, batch_task_ids, batch_shapes in dataloader:
    print(f"\nBatch shape: {batch_matrices.shape}")
    print(f"Data type: {batch_matrices.dtype}")
    print(f"Value range: {batch_matrices.min()} to {batch_matrices.max()}")
    print(f"Sample task IDs: {batch_task_ids[:3]}")
    print(f"Sample original shapes: {batch_shapes[:3]}")
    break

## 🔬 Experiment 1A: Embedding + CNN Encoder

**Hypothesis**: Learned embeddings can capture semantic relationships between colors while CNNs preserve spatial structure.

**Architecture**: 
- Embedding layer: integers 0-9 → 64D vectors
- CNN layers: 3 convolutional blocks with spatial processing
- Global pooling: Extract global features
- Decoder: Reconstruct original matrix for testing

In [ ]:
# Experiment 1A: Embedding + CNN Encoder
class EmbedCNNEncoder(nn.Module):
    """Embedding + CNN encoder for matrix reconstruction"""
    
    def __init__(self, embed_dim=64, hidden_dims=[128, 256, 512], output_dim=1024, max_size=30):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_size = max_size
        self.output_dim = output_dim
        
        # Color embedding: 0-9 -> embed_dim vectors
        self.embedding = nn.Embedding(10, embed_dim)
        
        # CNN layers for spatial processing
        self.conv_layers = nn.ModuleList()
        in_channels = embed_dim
        
        for hidden_dim in hidden_dims:
            self.conv_layers.append(nn.Sequential(
                nn.Conv2d(in_channels, hidden_dim, 3, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2)  # Reduce spatial dimensions
            ))
            in_channels = hidden_dim
        
        # Global feature extraction
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.feature_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim)
        )
        
        # Decoder for reconstruction
        self.decoder = self._build_decoder(hidden_dims)
    
    def _build_decoder(self, hidden_dims):
        """Build decoder to reconstruct original matrix"""
        # Simple decoder that upsamples and reconstructs
        decoder_layers = []
        
        # Start from global features
        decoder_layers.extend([
            nn.Linear(self.output_dim, hidden_dims[-1] * 4 * 4),
            nn.ReLU()
        ])
        
        # Reshape and upsample
        in_channels = hidden_dims[-1]
        for i, hidden_dim in enumerate(reversed(hidden_dims[:-1])):
            decoder_layers.extend([
                nn.ConvTranspose2d(in_channels, hidden_dim, 4, stride=2, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True)
            ])
            in_channels = hidden_dim
        
        # Final layer to output colors (10 classes)
        decoder_layers.append(nn.ConvTranspose2d(in_channels, 10, 4, stride=2, padding=1))
        
        return nn.Sequential(*decoder_layers)
    
    def forward(self, matrix):
        """Forward pass through encoder"""
        # matrix: (batch, height, width) with values 0-9
        batch_size, h, w = matrix.shape
        
        # Embed each cell: (batch, h, w, embed_dim)
        embedded = self.embedding(matrix)
        embedded = embedded.permute(0, 3, 1, 2)  # (batch, embed_dim, h, w)
        
        # Apply conv layers
        features = embedded
        spatial_features = []
        
        for conv_layer in self.conv_layers:
            features = conv_layer(features)
            spatial_features.append(features)
        
        # Global pooling
        global_features = self.global_pool(features).squeeze(-1).squeeze(-1)
        output_features = self.feature_head(global_features)
        
        return {
            'global_features': output_features,
            'spatial_features': spatial_features,
            'final_spatial': features
        }
    
    def decode(self, encoded_features):
        """Decode features back to matrix"""
        global_features = encoded_features['global_features']
        
        # Reshape to spatial format
        batch_size = global_features.size(0)
        x = self.decoder[0](global_features)  # Linear layer
        x = self.decoder[1](x)  # ReLU
        
        # Reshape to spatial
        x = x.view(batch_size, -1, 4, 4)
        
        # Apply transposed convolutions
        for layer in self.decoder[2:]:
            x = layer(x)
        
        # Interpolate to target size
        x = F.interpolate(x, size=(self.max_size, self.max_size), mode='bilinear', align_corners=False)
        
        return x
    
    def reconstruct(self, matrix):
        """Full reconstruction pipeline"""
        encoded = self.forward(matrix)
        decoded = self.decode(encoded)
        return decoded

# Initialize Experiment 1A model
embed_cnn_model = EmbedCNNEncoder().to(device)

# Count parameters
total_params = sum(p.numel() for p in embed_cnn_model.parameters())
trainable_params = sum(p.numel() for p in embed_cnn_model.parameters() if p.requires_grad)

print(f"EmbedCNN Model:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB")

# Test forward pass
with torch.no_grad():
    sample_batch, _, _ = next(iter(dataloader))
    sample_batch = sample_batch.to(device)
    
    print(f"\nTesting forward pass:")
    print(f"Input shape: {sample_batch.shape}")
    
    encoded = embed_cnn_model(sample_batch)
    print(f"Global features shape: {encoded['global_features'].shape}")
    print(f"Number of spatial feature maps: {len(encoded['spatial_features'])}")
    
    reconstructed = embed_cnn_model.decode(encoded)
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Reconstruction range: {reconstructed.min():.3f} to {reconstructed.max():.3f}")

In [ ]:
# Training setup for EmbedCNN
def train_encoder(model, dataloader, num_epochs=5, lr=0.001):
    """Train encoder for matrix reconstruction"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    training_losses = []
    
    print(f"Training {model.__class__.__name__} for {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, (matrices, task_ids, shapes) in enumerate(pbar):
            matrices = matrices.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            reconstructed = model.reconstruct(matrices)
            
            # Calculate loss on actual matrix regions
            total_loss = 0.0
            batch_size = matrices.size(0)
            
            # Debug: Print shapes structure for first batch
            if batch_idx == 0 and epoch == 0:
                print(f"Debug - shapes type: {type(shapes)}")
                print(f"Debug - shapes length: {len(shapes)}")
                print(f"Debug - first shape: {shapes[0]}")
                print(f"Debug - shape element type: {type(shapes[0])}")
            
            for i in range(batch_size):
                # Handle the tuple unpacking correctly
                if isinstance(shapes[i], tuple) and len(shapes[i]) == 2:
                    h, w = shapes[i]
                else:
                    # Fallback - shapes might be nested differently
                    shape_data = shapes[i]
                    if isinstance(shape_data, (list, tuple)) and len(shape_data) >= 2:
                        h, w = shape_data[0], shape_data[1]
                    else:
                        print(f"Warning: Unexpected shape format: {shape_data}")
                        continue
                
                h, w = int(h), int(w)  # Convert to int
                
                # Get original matrix region
                original = matrices[i, :h, :w]
                
                # Get reconstructed region and resize if needed
                recon = reconstructed[i, :, :h, :w]
                
                # Convert to class predictions
                recon_flat = recon.permute(1, 2, 0).reshape(-1, 10)  # (h*w, 10)
                original_flat = original.reshape(-1)  # (h*w,)
                
                # Calculate cross-entropy loss
                loss = criterion(recon_flat, original_flat)
                total_loss += loss
            
            # Average loss over batch
            avg_loss = total_loss / batch_size
            avg_loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_loss += avg_loss.item()
            num_batches += 1
            
            # Update progress bar
            pbar.set_postfix({
                'Loss': f'{avg_loss.item():.4f}',
                'Avg': f'{epoch_loss/num_batches:.4f}',
                'LR': f'{optimizer.param_groups[0]["lr"]:.6f}'
            })
            
            if batch_idx >= 100:  # Limit batches for quick testing
                break
        
        scheduler.step()
        
        avg_epoch_loss = epoch_loss / num_batches
        training_losses.append(avg_epoch_loss)
        
        print(f"Epoch {epoch+1}: Average Loss = {avg_epoch_loss:.4f}")
    
    return training_losses

# Train EmbedCNN model
print("=" * 60)
print("🚀 TRAINING EXPERIMENT 1A: EmbedCNN Encoder")
print("=" * 60)

embed_cnn_losses = train_encoder(embed_cnn_model, dataloader, num_epochs=3, lr=0.001)

## 🔬 Experiment 1B: One-Hot + CNN Encoder

**Goal**: Test one-hot encoding approach for matrix representation
- Convert each cell to one-hot vector (10 dimensions for colors 0-9)
- Use CNN to process spatial patterns
- Compare with embedding approach

**Architecture**:
- Input: Matrix (H×W) → One-hot (H×W×10)
- CNN: Multiple conv layers with pooling
- Output: Fixed-size feature vector

In [ ]:
# Experiment 1B: One-Hot + CNN Encoder
class OneHotCNNEncoder(nn.Module):
    """One-hot + CNN encoder for matrix reconstruction"""
    
    def __init__(self, hidden_dims=[64, 128, 256, 512], output_dim=1024, max_size=30):
        super().__init__()
        self.max_size = max_size
        self.output_dim = output_dim
        
        # CNN layers for spatial processing (input has 10 channels from one-hot)
        self.conv_layers = nn.ModuleList()
        in_channels = 10  # One-hot encoding has 10 channels
        
        for hidden_dim in hidden_dims:
            self.conv_layers.append(nn.Sequential(
                nn.Conv2d(in_channels, hidden_dim, 3, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2)  # Reduce spatial dimensions
            ))
            in_channels = hidden_dim
        
        # Global feature extraction
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.feature_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim)
        )
        
        # Decoder for reconstruction
        self.decoder = self._build_decoder(hidden_dims)
    
    def _build_decoder(self, hidden_dims):
        """Build decoder to reconstruct original matrix"""
        decoder_layers = []
        
        # Start from global features
        decoder_layers.extend([
            nn.Linear(self.output_dim, hidden_dims[-1] * 4 * 4),
            nn.ReLU()
        ])
        
        # Reshape and upsample
        in_channels = hidden_dims[-1]
        for i, hidden_dim in enumerate(reversed(hidden_dims[:-1])):
            decoder_layers.extend([
                nn.ConvTranspose2d(in_channels, hidden_dim, 4, stride=2, padding=1),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU(inplace=True)
            ])
            in_channels = hidden_dim
        
        # Final layer to output 10 channels (one-hot)
        decoder_layers.append(nn.ConvTranspose2d(in_channels, 10, 4, stride=2, padding=1))
        
        return nn.Sequential(*decoder_layers)
    
    def matrix_to_onehot(self, matrix):
        """Convert matrix to one-hot encoding"""
        batch_size, h, w = matrix.shape
        
        # Create one-hot encoding: (batch, h, w, 10)
        onehot = F.one_hot(matrix, num_classes=10).float()
        
        # Permute to (batch, 10, h, w) for CNN
        onehot = onehot.permute(0, 3, 1, 2)
        
        return onehot
    
    def forward(self, matrix):
        """Forward pass through encoder"""
        # Convert to one-hot: (batch, 10, h, w)
        onehot = self.matrix_to_onehot(matrix)
        
        # Apply conv layers
        features = onehot
        spatial_features = []
        
        for conv_layer in self.conv_layers:
            features = conv_layer(features)
            spatial_features.append(features)
        
        # Global pooling
        global_features = self.global_pool(features).squeeze(-1).squeeze(-1)
        output_features = self.feature_head(global_features)
        
        return {
            'global_features': output_features,
            'spatial_features': spatial_features,
            'final_spatial': features
        }
    
    def decode(self, encoded_features):
        """Decode features back to one-hot matrix"""
        global_features = encoded_features['global_features']
        
        # Reshape to spatial format
        batch_size = global_features.size(0)
        x = self.decoder[0](global_features)  # Linear layer
        x = self.decoder[1](x)  # ReLU
        
        # Reshape to spatial
        x = x.view(batch_size, -1, 4, 4)
        
        # Apply transposed convolutions
        for layer in self.decoder[2:]:
            x = layer(x)
        
        # Interpolate to target size
        x = F.interpolate(x, size=(self.max_size, self.max_size), mode='bilinear', align_corners=False)
        
        return x
    
    def reconstruct(self, matrix):
        """Full reconstruction pipeline"""
        encoded = self.forward(matrix)
        decoded = self.decode(encoded)
        return decoded

# Initialize Experiment 1B model
onehot_cnn_model = OneHotCNNEncoder().to(device)

# Count parameters
total_params = sum(p.numel() for p in onehot_cnn_model.parameters())
trainable_params = sum(p.numel() for p in onehot_cnn_model.parameters() if p.requires_grad)

print(f"OneHotCNN Model:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB")

# Test forward pass
with torch.no_grad():
    sample_batch, _, _ = next(iter(dataloader))
    sample_batch = sample_batch.to(device)
    
    print(f"\nTesting forward pass:")
    print(f"Input shape: {sample_batch.shape}")
    
    # Test one-hot conversion
    onehot = onehot_cnn_model.matrix_to_onehot(sample_batch)
    print(f"One-hot shape: {onehot.shape}")
    
    encoded = onehot_cnn_model(sample_batch)
    print(f"Global features shape: {encoded['global_features'].shape}")
    print(f"Number of spatial feature maps: {len(encoded['spatial_features'])}")
    
    reconstructed = onehot_cnn_model.decode(encoded)
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Reconstruction range: {reconstructed.min():.3f} to {reconstructed.max():.3f}")

In [ ]:
# Train OneHotCNN model
print("=" * 60)
print("🚀 TRAINING EXPERIMENT 1B: OneHotCNN Encoder")
print("=" * 60)

onehot_cnn_losses = train_encoder(onehot_cnn_model, dataloader, num_epochs=3, lr=0.001)

## 📊 Results Analysis & Visualization

**Evaluation Metrics**:
1. **Reconstruction Loss**: Cross-entropy between original and reconstructed matrices
2. **Pixel Accuracy**: Percentage of correctly reconstructed cells
3. **Feature Quality**: Visualization of learned representations
4. **Encoding Efficiency**: Feature vector quality and dimensionality

In [ ]:
# Evaluation and visualization functions
def evaluate_reconstruction(model, dataloader, num_samples=50):
    """Evaluate reconstruction quality"""
    model.eval()
    
    total_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    num_evaluated = 0
    
    criterion = nn.CrossEntropyLoss(reduction='none')
    
    with torch.no_grad():
        for batch_idx, (matrices, task_ids, shapes) in enumerate(dataloader):
            if num_evaluated >= num_samples:
                break
                
            matrices = matrices.to(device)
            batch_size = matrices.size(0)
            
            # Get reconstructions
            reconstructed = model.reconstruct(matrices)
            
            for i in range(min(batch_size, num_samples - num_evaluated)):
                h, w = shapes[i]  # shapes[i] is already a tuple (h, w)
                h, w = int(h), int(w)  # Convert to int
                
                # Get original matrix region
                original = matrices[i, :h, :w]
                
                # Get reconstructed region
                recon = reconstructed[i, :, :h, :w]
                
                # Convert to predictions
                recon_flat = recon.permute(1, 2, 0).reshape(-1, 10)  # (h*w, 10)
                original_flat = original.reshape(-1)  # (h*w,)
                
                # Calculate loss
                losses = criterion(recon_flat, original_flat)
                total_loss += losses.mean().item()
                
                # Calculate accuracy
                predictions = recon_flat.argmax(dim=1)
                correct_pixels += (predictions == original_flat).sum().item()
                total_pixels += original_flat.numel()
                
                num_evaluated += 1
                
                if num_evaluated >= num_samples:
                    break
    
    avg_loss = total_loss / num_evaluated
    accuracy = correct_pixels / total_pixels
    
    return {
        'avg_loss': avg_loss,
        'pixel_accuracy': accuracy,
        'total_samples': num_evaluated,
        'total_pixels': total_pixels
    }

def visualize_reconstructions(model, dataset, num_examples=6):
    """Visualize original vs reconstructed matrices"""
    model.eval()
    
    fig, axes = plt.subplots(2, num_examples, figsize=(15, 5))
    fig.suptitle(f'{model.__class__.__name__} Reconstructions', fontsize=14, fontweight='bold')
    
    with torch.no_grad():
        for i in range(num_examples):
            # Get random sample
            idx = np.random.randint(len(dataset))
            matrix, task_id, (h, w) = dataset[idx]  # Unpack correctly
            
            # Add batch dimension and move to device
            matrix_batch = matrix.unsqueeze(0).to(device)
            
            # Get reconstruction
            reconstructed = model.reconstruct(matrix_batch)[0]  # Remove batch dim
            
            # Convert to numpy for visualization
            original_np = matrix[:h, :w].cpu().numpy()
            
            # Get predicted classes
            recon_pred = reconstructed[:, :h, :w].argmax(dim=0).cpu().numpy()
            
            # Plot original
            axes[0, i].imshow(original_np, cmap='tab10', vmin=0, vmax=9)
            axes[0, i].set_title(f'Original #{idx}')
            axes[0, i].axis('off')
            
            # Plot reconstruction
            axes[1, i].imshow(recon_pred, cmap='tab10', vmin=0, vmax=9)
            axes[1, i].set_title('Reconstructed')
            axes[1, i].axis('off')
            
            # Calculate accuracy for this sample
            accuracy = (original_np == recon_pred).mean()
            axes[1, i].text(0.5, -0.1, f'Acc: {accuracy:.2f}', 
                          transform=axes[1, i].transAxes, ha='center')
    
    plt.tight_layout()
    plt.show()

def compare_models_performance():
    """Compare performance of different models"""
    print("=" * 60)
    print("📊 MODEL COMPARISON RESULTS")
    print("=" * 60)
    
    models = [
        ('EmbedCNN', embed_cnn_model),
        ('OneHotCNN', onehot_cnn_model)
    ]
    
    results = {}
    
    for name, model in models:
        print(f"\n🔍 Evaluating {name}...")
        eval_results = evaluate_reconstruction(model, dataloader, num_samples=100)
        results[name] = eval_results
        
        print(f"  📈 Results:")
        print(f"    • Average Loss: {eval_results['avg_loss']:.4f}")
        print(f"    • Pixel Accuracy: {eval_results['pixel_accuracy']:.3f}")
        print(f"    • Samples Evaluated: {eval_results['total_samples']}")
        print(f"    • Total Pixels: {eval_results['total_pixels']:,}")
    
    # Create comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    model_names = list(results.keys())
    losses = [results[name]['avg_loss'] for name in model_names]
    accuracies = [results[name]['pixel_accuracy'] for name in model_names]
    
    # Loss comparison
    bars1 = ax1.bar(model_names, losses, color=['skyblue', 'lightcoral'])
    ax1.set_title('Average Reconstruction Loss')
    ax1.set_ylabel('Cross-Entropy Loss')
    for i, bar in enumerate(bars1):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{losses[i]:.3f}', ha='center', va='bottom')
    
    # Accuracy comparison
    bars2 = ax2.bar(model_names, accuracies, color=['lightgreen', 'orange'])
    ax2.set_title('Pixel Accuracy')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    for i, bar in enumerate(bars2):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{accuracies[i]:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return results

def plot_training_curves():
    """Plot training loss curves"""
    plt.figure(figsize=(10, 6))
    
    if 'embed_cnn_losses' in globals():
        plt.plot(embed_cnn_losses, 'b-', label='EmbedCNN', linewidth=2)
    
    if 'onehot_cnn_losses' in globals():
        plt.plot(onehot_cnn_losses, 'r-', label='OneHotCNN', linewidth=2)
    
    plt.title('Training Loss Curves', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Run evaluations
print("🚀 Starting comprehensive evaluation...")
comparison_results = compare_models_performance()

# Visualize reconstructions
print("\n🖼️  Visualizing reconstructions...")
for name, model in [('EmbedCNN', embed_cnn_model), ('OneHotCNN', onehot_cnn_model)]:
    visualize_reconstructions(model, dataset, num_examples=6)

# Plot training curves
print("\n📈 Plotting training curves...")
plot_training_curves()

## 🎯 Stage 1 Conclusions & Next Steps

### Summary of Experiments

**Stage 1 Goal**: Test basic matrix encoding approaches for ARC data

**Experiments Completed**:
1. **Experiment 1A - EmbedCNN**: Embedding + CNN encoder for matrix representation
2. **Experiment 1B - OneHotCNN**: One-hot + CNN encoder for matrix representation

### Key Findings

Based on the reconstruction experiments, we can draw conclusions about:

1. **Encoding Effectiveness**: Which approach better captures matrix patterns?
2. **Parameter Efficiency**: Which model achieves good performance with fewer parameters?
3. **Reconstruction Quality**: How well do models preserve spatial relationships?
4. **Training Stability**: Which approach converges more reliably?

### Decision Points for Stage 2

**If EmbedCNN performs better**:
- ✅ Proceed with embedding-based encoders
- 🔄 Experiment with different embedding dimensions
- 🚀 Move to Stage 2: Dual encoder architecture

**If OneHotCNN performs better**:
- ✅ Proceed with one-hot encoding
- 🔄 Experiment with different CNN architectures  
- 🚀 Move to Stage 2: Dual encoder architecture

**If both perform similarly**:
- 🤔 Try hybrid approach combining both methods
- 📊 Use ensemble of both encoders
- 🔬 Investigate additional encoder types (Transformer, etc.)

### Next Stage Preview

**Stage 2**: Dual Encoder Architecture
- Build separate input and output encoders using best approach from Stage 1
- Train encoders to learn complementary representations
- Implement central "brain" to process encoded features
- Test on input→output transformation tasks

In [ ]:
# Final summary and model saving
def save_experiment_results():
    """Save model states and experiment results"""
    results_summary = {
        'experiment': 'Stage1_Matrix_Encoding',
        'date': '2025-01-04',
        'models_tested': ['EmbedCNN', 'OneHotCNN'],
        'dataset_size': len(dataset),
        'device': str(device),
    }
    
    # Add model parameters info
    for model_name, model in [('EmbedCNN', embed_cnn_model), ('OneHotCNN', onehot_cnn_model)]:
        total_params = sum(p.numel() for p in model.parameters())
        results_summary[f'{model_name}_params'] = total_params
    
    # Add performance results if available
    if 'comparison_results' in globals():
        results_summary['performance_results'] = comparison_results
    
    print("=" * 60)
    print("💾 EXPERIMENT SUMMARY")
    print("=" * 60)
    
    for key, value in results_summary.items():
        if key != 'performance_results':
            print(f"📋 {key}: {value}")
    
    if 'comparison_results' in globals():
        print("\n📊 Performance Summary:")
        for model_name, results in comparison_results.items():
            print(f"  🔹 {model_name}:")
            print(f"    • Loss: {results['avg_loss']:.4f}")
            print(f"    • Accuracy: {results['pixel_accuracy']:.3f}")
    
    print("\n✅ Stage 1 Complete! Ready for Stage 2: Dual Encoder Architecture")
    
    return results_summary

# Execute final summary
experiment_summary = save_experiment_results()

print("\n" + "="*60)
print("🎉 STAGE 1 MATRIX ENCODING EXPERIMENTS COMPLETE!")
print("="*60)
print("\n📋 Experiment Status: ✅ SUCCESS")
print("🚀 Next Step: Implement Stage 2 - Dual Encoder Architecture")
print("📊 Data: All models trained and evaluated on ARC dataset")
print("💡 Insights: Ready to build input/output encoder pairs")

print("\n🔗 Continue with Stage 2 implementation using best performing encoder architecture!")